# Chaînes-05 — L'énergie et l'Hamiltonien : l'opérateur à première puissance et le pas

Cinquième carnet de la série [Chaînes](README.md) — distillation du chapitre 44 du compendium Chain Theory
(Wissner-Gross, 2026, preprint non relu) : *Energy and the Hamiltonian: the first-power operator
and the step*. Le fil : après les corps qu'on ne peut pas distinguer (Chaînes-04), ce que la
construction lit comme **l'énergie** — la somme de chaîne à première puissance, l'objet que la
littérature appelle l'Hamiltonien apparaissant ici comme **le générateur du pas lu à la première
puissance** — et l'identification de l'énergie et de la masse par le **carré de Born du niveau
du corps**.

Patron de la série : **Mesuré** (rejoué par le code de ce carnet) · **Cité** (toute la chaîne
dérivative de la source, jamais rejouée ici) · **Ouvert** (ce que la source ne prouve pas,
listé ligne à ligne, jamais maquillé en résultat établi).

**Navigation** : [<< Chaînes-04 — L'indiscernabilité](Chaines-04-Indiscernabilite-Spin-Statistique-Python.ipynb) | [Index de la série](README.md) | [Version Python](Chaines-05-Energie-Hamiltonien-Premiere-Puissance-Python.ipynb) | [Chaînes-06 — L'intégrale de chemin >>](Chaines-06-Integrale-Chemin-Somme-Finie-Enregistrements-CSharp.ipynb)

In [1]:
#nullable enable
using System;
using System.Globalization;
using System.Numerics;

// Exact rational arithmetic : reduced fraction of BigInteger (numerator/denominator).
public sealed class Rat
{
    public BigInteger Num { get; }
    public BigInteger Den { get; }

    public Rat(BigInteger num, BigInteger den)
    {
        if (den == BigInteger.Zero)
        {
            throw new DivideByZeroException("denominateur nul");
        }
        BigInteger g = BigInteger.GreatestCommonDivisor(BigInteger.Abs(num), BigInteger.Abs(den));
        if (g == BigInteger.Zero) { g = BigInteger.One; }
        BigInteger n = num / g;
        BigInteger d = den / g;
        if (d < BigInteger.Zero) { n = -n; d = -d; }
        Num = n;
        Den = d;
    }

    public static Rat Make(BigInteger num, BigInteger den) => new Rat(num, den);

    public Rat AuCarre() => new Rat(Num * Num, Den * Den);

    public static Rat operator *(Rat a, Rat b) => new Rat(a.Num * b.Num, a.Den * b.Den);

    public static bool operator ==(Rat a, Rat b)
    {
        if (a is null) { return b is null; }
        if (b is null) { return false; }
        return a.Num == b.Num && a.Den == b.Den;
    }

    public static bool operator !=(Rat a, Rat b) => !(a == b);

    public override bool Equals(object? obj) => obj is Rat r && this == r;

    public override int GetHashCode() => Num.GetHashCode() ^ Den.GetHashCode();

    public override string ToString()
    {
        string n = Num.ToString(CultureInfo.InvariantCulture);
        if (Den == BigInteger.One) { return n; }
        return n + "/" + Den.ToString(CultureInfo.InvariantCulture);
    }
}

// Executable assertions : they raise when a computed value differs from the expected one.
public static class Verif
{
    public static void Egale(object obtenu, object attendu, string libelle)
    {
        bool conforme = object.Equals(obtenu, attendu);
        Console.WriteLine((conforme ? "CONFORME  " : "ECART     ") + libelle
            + " : obtenu " + obtenu + " | attendu " + attendu);
        if (!conforme)
        {
            throw new InvalidOperationException("Ecart sur " + libelle
                + " : obtenu " + obtenu + ", attendu " + attendu);
        }
    }
}


## Le cadre : lire l'énergie comme la somme à première puissance

Le théorème 44.1 est **conditionnel** — `Standing: a conditional theorem` — posé sur le texte
de l'axiome et sur les résultats des chapitres nommés en prémices (Définitions 1.1-1.6,
Théorèmes 18.1-18.4, résultats ouverts 18.5-18.6, Théorème 40.1, résultats ouverts et théorèmes
241.x et 245.x — la chaîne de prémices la plus longue du compendium remonte à l'axiome par
**douze chapitres**). Sous cette lecture :

- la **somme de chaîne à première puissance** est un opérateur sur l'espace des enregistrements
  de chacune des **309 lectures tabulées**, avec la somme à première puissance de la toile comme
  valeur propre — *census, cité plus bas* ;
- elle **commute avec le pas**, dont le générateur (le générateur du pas) se retrouve à chaque
  type de paire à un niveau et se conserve aux **32851 coups** du census ;
- le générateur produit porte la somme à première puissance comme niveau, et le **carré de Born
  du niveau du corps** égale l'énergie de repos debout aux **treize corps** — la forme de
Schrödinger tient à l'enregistrement comme composition de deux objets établis.

Ce carnet rejoue les quatre lignes exactes (64)-(67) que la source déroule au cas le plus petit,
plus l'arithmétique qu'elle invite le lecteur à reprendre à la main. Le théorème lui-même, son
census et ses deux moteurs restent **cités**.


## 1. La somme à première puissance d'une toile (éq. 64)

La partie exacte de la dérivation commence par la **mesure d'une toile et son produit hub** : de
la mesure de la toile et de son produit hub, par la règle de première puissance **sans rien
frapper**, on obtient la somme à première puissance de la toile. Au **cas le plus petit du
catalogue** — la lecture de la toile de mesure **116** à ses deux sites qubits — le produit hub
est **36**, et la somme à première puissance est :

$$S_0 = 116 \cdot 36 = 4176. \qquad (64)$$

La règle qui multiplie la mesure par le produit hub appartient à la source (règle de
première puissance) ; ce que le code vérifie est l'arithmétique annoncée.


In [2]:
using System;
using System.Globalization;
using System.Numerics;

// Eq. 64 : somme a premiere puissance de la plus petite lecture du catalogue.
BigInteger mesure_toile = 116;   // la toile de mesure 116 a ses deux sites qubits (plus petite lecture)
BigInteger produit_hub = 36;     // produit hub de cette toile (CITE)
BigInteger S0 = mesure_toile * produit_hub;

Console.WriteLine("mesure de la toile : " + mesure_toile.ToString(CultureInfo.InvariantCulture) + " (les deux sites qubits)");
Console.WriteLine("produit hub        : " + produit_hub.ToString(CultureInfo.InvariantCulture) + " (cite)");
Console.WriteLine("S0 = 116 * 36      = " + S0.ToString(CultureInfo.InvariantCulture) + " (eq. 64 annonce 4176)");
Verif.Egale(S0, new BigInteger(4176), "eq. 64 : S0 = 116 * 36");


mesure de la toile : 116 (les deux sites qubits)


produit hub        : 36 (cite)


S0 = 116 * 36      = 4176 (eq. 64 annonce 4176)


CONFORME  eq. 64 : S0 = 116 * 36 : obtenu 4176 | attendu 4176


### Lecture du résultat

L'arithmétique se referme : $116 \cdot 36 = 4176$, exactement l'éq. 64. La mesure 116 et le
produit hub 36 sont **donnés** par la source (ils viennent de la construction de la toile, hors
du périmètre de ce carnet) — ce que le code prouve est que la règle de première puissance,
appliquée à ces deux nombres, rend bien la somme annoncée. Le nombre 4176 va revenir à la
ligne (66) comme **valeur propre** de l'opérateur à première puissance sur l'espace des
enregistrements.


## 2. L'espace des enregistrements à la lecture slot (éq. 65)

De l'espace des enregistrements **de cette même lecture** à la lecture slot, par l'opérateur de
gain au niveau enregistrement, la source obtient **cinquante-sept enregistrements** avec les
gains **un à vingt-quatre, deux à trente et quatre à trois**, la trace de l'opérateur de gain
un élément exact de l'anneau égal à sa forme fermée, et la **face de Born 96** :

$$\operatorname{tr} G = 30 + 30\sqrt{2}, \qquad Z = 96, \qquad N = 24 + 30 + 3 = 57. \qquad (65)$$

Deux faits à garder séparés : la **trace** $\operatorname{tr} G = 30 + 30\sqrt2$ vit dans
l'anneau (pondération d'anneau, pas une somme d'entiers) et reste **citée** ; la **somme des
gains entiers** $24 \cdot 1 + 30 \cdot 2 + 3 \cdot 4$, elle, se calcule — et tombe sur $Z = 96$,
la face de Born. Et un carrefour avec Chaînes-03 : l'éq. 58 (ch. 42) dénombrait **le même**
espace par secteurs — $24 + 18 + 12 + 3 = 57$ — deux décompositions du même espace
d'enregistrements, la face de Born restant 96 dans les deux chapitres.


In [3]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

// Eq. 65 : l'espace des enregistrements a la lecture slot (gains CITES, additions reverifiees).
var gains = new Dictionary<BigInteger, BigInteger>
{
    { new BigInteger(1), new BigInteger(24) },   // gain -> nombre d'enregistrements (cites)
    { new BigInteger(2), new BigInteger(30) },
    { new BigInteger(4), new BigInteger(3) }
};
BigInteger N = 0;                              // nombre d'enregistrements
foreach (var kv in gains.OrderBy(k => k.Key))
{
    N += kv.Value;
    Console.WriteLine("gain " + kv.Key.ToString(CultureInfo.InvariantCulture)
        + " : " + kv.Value.ToString(CultureInfo.InvariantCulture) + " enregistrements");
}
BigInteger somme_gains = 0;                    // somme des gains entiers
foreach (var kv in gains)
{
    somme_gains += kv.Key * kv.Value;
}
BigInteger Z = new BigInteger(96);             // face de Born (citee)
Console.WriteLine("N = 24 + 30 + 3                = " + N.ToString(CultureInfo.InvariantCulture) + " (eq. 65 annonce 57)");
Console.WriteLine("somme des gains entiers        = " + somme_gains.ToString(CultureInfo.InvariantCulture)
    + " = Z = " + Z.ToString(CultureInfo.InvariantCulture));
Console.WriteLine("tr G = 30 + 30*sqrt(2)         (citee, element de l'anneau)");
Verif.Egale(N, new BigInteger(57), "eq. 65 : nombre d'enregistrements");
Verif.Egale(somme_gains, Z, "eq. 65 : somme des gains entiers = face de Born");

// Carrefour avec Chaines-03 (eq. 58, ch. 42) : le meme espace compte par secteurs.
var secteurs_eq58 = new Dictionary<string, BigInteger>
{
    { "impairs", new BigInteger(24) },
    { "pairs mixtes", new BigInteger(18) },
    { "pairs identiques", new BigInteger(12) },
    { "cloture", new BigInteger(3) }
};
BigInteger N_eq58 = 0;
foreach (var kv in secteurs_eq58)
{
    N_eq58 += kv.Value;
}
Console.WriteLine("eq. 58 (ch. 42) : 24+18+12+3   = " + N_eq58.ToString(CultureInfo.InvariantCulture));
Verif.Egale(N, N_eq58, "carrefour eq. 58 : meme espace, deux decompositions");

gain 1 : 24 enregistrements


gain 2 : 30 enregistrements


gain 4 : 3 enregistrements


N = 24 + 30 + 3                = 57 (eq. 65 annonce 57)


somme des gains entiers        = 96 = Z = 96


tr G = 30 + 30*sqrt(2)         (citee, element de l'anneau)


CONFORME  eq. 65 : nombre d'enregistrements : obtenu 57 | attendu 57


CONFORME  eq. 65 : somme des gains entiers = face de Born : obtenu 96 | attendu 96


eq. 58 (ch. 42) : 24+18+12+3   = 57


CONFORME  carrefour eq. 58 : meme espace, deux decompositions : obtenu 57 | attendu 57


### Lecture du résultat

Les deux additions se referment. D'un côté $24 + 30 + 3 = 57$ enregistrements, comme l'énonce
l'éq. 65 ; de l'autre, la **somme des gains entiers** $24 \cdot 1 + 30 \cdot 2 + 3 \cdot 4 = 96$
retombe exactement sur la face de Born $Z = 96$ — un fait d'arithmétique que la source énonce et
que ce code vérifie, **pas** une dérivation de $Z$ (la face de Born se construit aux chapitres 18
et 28, cités). Le carrefour est aussi vérifié : l'éq. 58 de Chaînes-03 comptait le même espace
en secteurs $24 + 18 + 12 + 3$, même total 57. La trace $\operatorname{tr} G = 30 + 30\sqrt2$
reste citée : elle vit dans l'anneau, sa construction n'est pas rejouée ici.


## 3. La valeur propre, le commutateur et les niveaux (éq. 66)

De l'opérateur à première puissance **à cette lecture**, par sa valeur propre sur l'espace des
enregistrements, la source obtient la somme à première puissance de la toile comme **valeur
propre de multiplicité cinquante-sept** — autant que d'enregistrements —, son **commutateur avec
le générateur du pas nul aux douze tests de commutation**, et son niveau à l'**unité hub** et à
l'**unité dock** comme les rationnels exacts :

$$H\psi = 4176\,\psi, \qquad [H, G_{\text{step}}] = 0, \qquad \ell_{\text{hub}} = \frac{1392}{67}, \qquad \ell_{\text{dock}} = \frac{4176}{67}. \qquad (66)$$

Deux décodings que le texte du PDF écrase en « 139267 » et « 417667 » : le niveau hub se prend
en divisant la somme par l'**habillage hub 3** puis par le **rang 67** ($4176/3 = 1392$, puis
$1392/67$) ; le niveau dock divise directement par le rang ($4176/67$). La construction de
l'opérateur $H$, celle du générateur du pas $G_{\text{step}}$ et les douze tests sont **cités**
— ce que le code vérifie est la cohérence des niveaux annoncés avec la somme 4176, l'habillage
et le rang.


In [4]:
using System;
using System.Globalization;
using System.Numerics;

// Eq. 66 : valeur propre, commutateur (CITE) et niveaux rationnels exacts.
BigInteger valeur_propre = S0;                    // valeur propre reprise du calcul de l'eq. 64
BigInteger multiplicite = N;                     // multiplicite reprise du calcul de l'eq. 65
BigInteger habillage_hub = new BigInteger(3);     // habillage hub (cite)
BigInteger rang = new BigInteger(67);             // rang (cite)

// Division intermediaire replayee comme dans la source, puis niveau rationnel exact.
BigInteger quotient_hub = valeur_propre / habillage_hub;
BigInteger reste_hub = valeur_propre % habillage_hub;
Rat ell_hub = Rat.Make(quotient_hub, rang);
Rat ell_dock = Rat.Make(valeur_propre, rang);

Console.WriteLine("H psi = " + valeur_propre.ToString(CultureInfo.InvariantCulture)
    + " psi, multiplicite = " + multiplicite.ToString(CultureInfo.InvariantCulture)
    + " enregistrements (cite : 57)");
Console.WriteLine("[H, G_step] = 0 aux 12 tests de commutation (CITE -- construction du pas = source)");
Console.WriteLine("division de S0 par l'habillage hub : quotient = "
    + quotient_hub.ToString(CultureInfo.InvariantCulture)
    + ", reste = " + reste_hub.ToString(CultureInfo.InvariantCulture));
Console.WriteLine("niveau a l'unite hub   : (" + quotient_hub.ToString(CultureInfo.InvariantCulture)
    + "/" + rang.ToString(CultureInfo.InvariantCulture) + ") = " + ell_hub.ToString());
Console.WriteLine("niveau a l'unite dock  : " + valeur_propre.ToString(CultureInfo.InvariantCulture)
    + "/" + rang.ToString(CultureInfo.InvariantCulture) + "     = " + ell_dock.ToString());
Verif.Egale(valeur_propre, new BigInteger(4176), "eq. 66 : valeur propre reprise de l'eq. 64");
Verif.Egale(multiplicite, new BigInteger(57), "eq. 66 : multiplicite reprise de l'eq. 65");
Verif.Egale(quotient_hub, new BigInteger(1392), "eq. 66 : quotient de S0 par l'habillage hub");
Verif.Egale(reste_hub, BigInteger.Zero, "eq. 66 : reste nul de la division par l'habillage hub");
Verif.Egale(ell_hub, Rat.Make(1392, 67), "eq. 66 : niveau a l'unite hub");
Verif.Egale(ell_dock, Rat.Make(4176, 67), "eq. 66 : niveau a l'unite dock");
// Decoding des fractions ecrasees du PDF : "139267" -> 1392/67, "417667" -> 4176/67.
Verif.Egale(Rat.Make(1392, 67), ell_hub, "decoding PDF '139267' = niveau hub");
Verif.Egale(Rat.Make(4176, 67), ell_dock, "decoding PDF '417667' = niveau dock");


H psi = 4176 psi, multiplicite = 57 enregistrements (cite : 57)


[H, G_step] = 0 aux 12 tests de commutation (CITE -- construction du pas = source)


division de S0 par l'habillage hub : quotient = 1392, reste = 0


niveau a l'unite hub   : (1392/67) = 1392/67


niveau a l'unite dock  : 4176/67     = 4176/67


CONFORME  eq. 66 : valeur propre reprise de l'eq. 64 : obtenu 4176 | attendu 4176


... (3 outputs left out)

CONFORME  eq. 66 : niveau a l'unite hub : obtenu 1392/67 | attendu 1392/67


CONFORME  eq. 66 : niveau a l'unite dock : obtenu 4176/67 | attendu 4176/67


CONFORME  decoding PDF '139267' = niveau hub : obtenu 1392/67 | attendu 1392/67


CONFORME  decoding PDF '417667' = niveau dock : obtenu 4176/67 | attendu 4176/67


### Lecture du résultat

Les niveaux sont des **rationnels exacts** et les deux décodings tiennent : le niveau hub
$(4176/3)/67 = 1392/67$ et le niveau dock $4176/67$ sont exactement les fractions que le texte
du PDF écrase en « 139267 » et « 417667 ». La multiplicité 57 n'est pas un hasard : l'espace
des enregistrements de l'éq. 65 a 57 dimensions, et l'opérateur à première puissance y porte
la somme $4176$ **sur chaque enregistrement** — valeur propre de multiplicité 57. Ce que ce
carnet ne rejoue pas : la construction de l'opérateur $H$, celle du générateur du pas et les
douze tests de commutation — le commutateur nul $[H, G_{\text{step}}] = 0$ est **cité**, et il
est le cœur physique du chapitre (l'énergie commute avec le pas : la quantité conservée).


## Exercice 1 — Une autre lecture, une autre somme

La règle de première puissance (éq. 64) ne dépend pas de la taille de la toile : somme à
première puissance $=$ mesure de la toile $\times$ produit hub. Une lecture hypothétique du
catalogue porte une mesure de $140$ et un produit hub de $12$.

**Consigne.** Calculez la somme à première puissance de cette lecture, puis ses niveaux à
l'unité hub et à l'unité dock **pour un rang de 12** (habillage hub $2$), en rationnels exacts
(le type `Rat` de la cellule 1).


In [5]:
// Exercice 1 : a vous -- somme a premiere puissance et niveaux d'une lecture hypothetique.
public static class Exercice1
{
    public static void Resoudre()
    {
        // TODO etudiant
        Console.WriteLine("Exercice a completer");
    }
}

Exercice1.Resoudre();

Exercice a completer


## 4. Le corps électron et l'énergie de repos (éq. 67, fig. 19)

Du **corps électron à ses deux docks** (figure 19), par la règle du corps, la source obtient le
**rang dock-frappé 67**, l'**habillage hub 3** et le **niveau du corps 201** :

- **sites** : deux qutrits $B$, $C$ (dimension 3) et deux qubits $w_1$, $w_2$ (dimension 2) ;
- **lien cœur** $B$–$C$ de type $(3,3)$ ; **liens dock** $B$–$w_1$ et $C$–$w_2$ de type $(2,3)$ —
  les deux liens **frappés** de la règle du corps ;
- niveau du corps $S = 67 \cdot 3 = 201$, dont le **carré de Born** à l'unité hub est **1** et à
  l'unité dock est **9** :

$$S = 67 \cdot 3 = 201, \qquad \ell_{\text{hub}} = 1, \quad \ell_{\text{dock}} = 3, \qquad \ell^2_{\text{hub}} = 1, \qquad \ell^2_{\text{dock}} = 9. \qquad (67)$$

C'est le moment $E = mc^2$ du chapitre : le carré de Born du niveau du corps **égale l'énergie
de repos debout aux deux unités** — la correspondance [78] nomme l'énergie de repos d'un corps
sa masse à l'unité où la vitesse de la lumière est une. La correspondance est **par nom**.


In [6]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

// Eq. 67 + figure 19 : le corps electron a ses deux docks (rang et habillage CITES).
var dims_corps = new Dictionary<string, BigInteger>
{
    { "B", new BigInteger(3) },
    { "C", new BigInteger(3) },
    { "w1", new BigInteger(2) },
    { "w2", new BigInteger(2) }
};   // deux qutrits, deux qubits
var liens_corps = new Dictionary<string, (string a, string b, BigInteger ta, BigInteger tb)>
{
    { "coeur B-C", ("B", "C", new BigInteger(3), new BigInteger(3)) },   // lien coeur (3,3)
    { "dock B-w1", ("B", "w1", new BigInteger(3), new BigInteger(2)) },  // lien dock frappe (2,3)
    { "dock C-w2", ("C", "w2", new BigInteger(3), new BigInteger(2)) }   // lien dock frappe (2,3)
};
BigInteger rang_corps = new BigInteger(67);       // rang dock-frappe (cite)
BigInteger habillage_corps = new BigInteger(3);   // habillage hub (cite)
BigInteger S = rang_corps * habillage_corps;   // niveau du corps
Rat ell_hub = Rat.Make(S, habillage_corps * rang_corps);
Rat ell_dock = Rat.Make(S, rang_corps);
Rat ell2_hub = ell_hub.AuCarre();     // carre de Born a l'unite hub
Rat ell2_dock = ell_dock.AuCarre();   // carre de Born a l'unite dock

// Coherence structurelle : chaque type de lien accorde les dimensions de ses extremites.
foreach (var kv in liens_corps)
{
    var lien = kv.Value;
    bool coherent = new[] { lien.ta, lien.tb }.OrderBy(x => x)
        .SequenceEqual(new[] { dims_corps[lien.a], dims_corps[lien.b] }.OrderBy(x => x));
    BigInteger type_min = lien.ta < lien.tb ? lien.ta : lien.tb;
    BigInteger type_max = lien.ta < lien.tb ? lien.tb : lien.ta;
    Console.WriteLine("lien " + kv.Key.PadRight(12) + " type (" + type_min.ToString(CultureInfo.InvariantCulture) + "," + type_max.ToString(CultureInfo.InvariantCulture) + ") "
        + "extremites " + dims_corps[lien.a].ToString(CultureInfo.InvariantCulture) + "/" + dims_corps[lien.b].ToString(CultureInfo.InvariantCulture)
        + " -> coherent : " + coherent.ToString());
    Verif.Egale(coherent, true, "coherence du lien " + kv.Key);
}
Console.WriteLine("S = 67 * 3                     = " + S.ToString(CultureInfo.InvariantCulture) + " (eq. 67 annonce 201)");
Console.WriteLine("niveau a l'unite hub   : 201/(3*67) = " + ell_hub.ToString() + " et carre de Born = " + ell2_hub.ToString());
Console.WriteLine("niveau a l'unite dock  : 201/67     = " + ell_dock.ToString() + " et carre de Born = " + ell2_dock.ToString());
Console.WriteLine("energie de repos debout egalee aux deux unites (correspondance [78], PAR NOM)");
Verif.Egale(S, new BigInteger(201), "eq. 67 : niveau du corps S = 67 * 3");
Verif.Egale(ell_hub, Rat.Make(1, 1), "eq. 67 : niveau a l'unite hub");
Verif.Egale(ell_dock, Rat.Make(3, 1), "eq. 67 : niveau a l'unite dock");
Verif.Egale(ell2_hub, Rat.Make(1, 1), "eq. 67 : carre de Born a l'unite hub");
Verif.Egale(ell2_dock, Rat.Make(9, 1), "eq. 67 : carre de Born a l'unite dock");

lien coeur B-C    type (3,3) extremites 3/3 -> coherent : True


CONFORME  coherence du lien coeur B-C : obtenu True | attendu True


lien dock B-w1    type (2,3) extremites 3/2 -> coherent : True


CONFORME  coherence du lien dock B-w1 : obtenu True | attendu True


lien dock C-w2    type (2,3) extremites 3/2 -> coherent : True


CONFORME  coherence du lien dock C-w2 : obtenu True | attendu True


... (5 outputs left out)

CONFORME  eq. 67 : niveau a l'unite hub : obtenu 1 | attendu 1


CONFORME  eq. 67 : niveau a l'unite dock : obtenu 3 | attendu 3


CONFORME  eq. 67 : carre de Born a l'unite hub : obtenu 1 | attendu 1


CONFORME  eq. 67 : carre de Born a l'unite dock : obtenu 9 | attendu 9


### Lecture du résultat

Les trois types de liens sont structurellement cohérents avec les dimensions des sites
($(3,3)$ entre les deux qutrits, $(2,3)$ entre chaque qubit et son qutrit), et l'arithmétique
se referme exactement : $S = 67 \cdot 3 = 201$, niveau **1** à l'unité hub et **3** à l'unité
dock, carrés de Born **1** et **9** — des rationnels qui tombent juste, et c'est le fait : le
carré de Born du niveau du corps égale **l'énergie de repos debout aux deux unités**. Le rang
dock-frappé 67 et l'habillage hub 3 restent **cités** (ils viennent de la règle du corps, dont
la construction appartient à la source) ; la correspondance avec $E = mc^2$ est **par nom**
([78]) : le compendium fournit un objet fini de cette théorie qui porte le nom, pas une
dérivation de la relativité.


## Exercice 2 — Un second corps, l'additivité en rang

La source cite l'énergie « exacte aux neuf corps et aux 255 jointures tabulées, additive en
rang avec l'habillage en facteur commun aux 43 jointures non interactives » — la règle du
niveau $S = \text{rang} \cdot \text{habillage}$, appliquée au corps électron, vaut pour tous les
corps du census.

**Consigne.** Un corps hypothétique porte un rang dock-frappé de $134$ (le double du corps
électron) et le même habillage hub $3$. Calculez son niveau, ses niveaux aux deux unités et
les carrés de Born correspondants, en rationnels exacts (le type `Rat` de la cellule 1). Que vaut le carré de Born à l'unité dock, et dépend-t-il du rang ?


In [7]:
// Exercice 2 : a vous -- niveau et carres de Born d'un corps de rang 134.
public static class Exercice2
{
    public static void Resoudre()
    {
        // TODO etudiant
        Console.WriteLine("Exercice a completer");
    }
}

Exercice2.Resoudre();

Exercice a completer


## 5. Le census — tout ce que ce carnet ne rejoue pas (**Cité**)

La partie census de la dérivation est une énumération exhaustive que ce carnet **ne rejoue
pas** ; chaque compte est cité tel quel :

- **309 lectures tabulées** du catalogue : l'opérateur à première puissance y est construit à
  chaque lecture depuis l'opérateur de gain, sa valeur propre égale à la somme à première
  puissance de la toile **à chaque lecture** ; le commutateur avec le générateur du pas y est
  nul **à chaque lecture** ; le générateur produit y porte la somme comme niveau **à chaque
  lecture** ;
- le générateur du pas se retrouve aux **six types de paires à un niveau**, son fonctionnel
  central conservé aux **32851 coups** du census ;
- le carré de Born du niveau du corps égale l'énergie de repos debout aux **treize corps, aux
  deux unités** ;
- l'énergie d'un corps est « exacte aux **neuf corps** et aux **255 jointures tabulées**,
  additive en rang avec l'habillage en facteur commun aux **43 jointures non interactives** » ;
- aux **69211 registres des 2276 toiles et puits** tabulées, le générateur du pas égale
  l'opérateur d'ordre à l'unité debout, « exactement, ce qui dissout la question de
  l'opérateur d'ordre » ;
- l'opérateur de transfert des flèches satisfait l'identité de renversement, ses **4080 cycles
  premiers** au cap portant un argument discret à face réelle — zéro aux cycles pairs, un
  demi-tour aux cycles impairs de support à deux bases ;
- à une étoile par un centre de degré trois, les **169 enregistrements**, leur split
  **79/48/42** et la clôture **3** se retrouvent exactement sous la règle d'enregistrement
  (théorème conditionnel dont l'autre face est ouverte) ; l'identité site-déterminant tient aux
  **64 toiles** de la construction ;
- **contre-tests** : les objets altérés un à un, chaque lecture à sa lettre épinglée aux deux
  moteurs ; **contrôles par hash** (toiles, corps, lectures, résultat de signe, résultat de
  paire plate, résultat d'énergie de repos relus à l'identique aux deux moteurs) ;
  **second moteur** construit à part, les deux runs égaux à chaque clé ;
- le paquet vérifié par machine nommé à l'appendice de reproductivité certifie les énoncés du
  chapitre sous les définitions du compendium « et ne prouve aucun théorème de la
  littérature » — *32851 objects enumerated*.


## 6. Ce que la source ne prouve pas (**Ouvert**)

Trois résultats ouverts se tiennent dans ce chapitre, énoncés ligne à ligne :

1. **Le générateur debout n'est pas la somme à première puissance** : le générateur debout —
   une phase globale à un niveau — égale la somme à première puissance à l'unité **à aucune des
   309 lectures**. L'égalité de la forme de Schrödinger tient à l'enregistrement comme
   composition de deux objets établis ; l'identification directe, elle, échoue à chaque
   lecture.
2. **Le transport debout part de l'exponentielle** : le transport debout **part** de
   l'exponentielle de l'opérateur à première puissance à un tick aux **1692 liens** —
   l'opérateur à première puissance n'engendre qu'une phase ; des cinq classes de corollaires
   voisines, **deux sont dérivées et trois sont des glosses**.
3. **Le spectre de l'exposant part du spectre d'ordre** : le spectre de l'exposant du
   générateur de transport **part** du spectre d'ordre debout aux **1692 liens** — un parting
   de dimension, dissous au second résultat de sa ligne.

**Prémices ouvertes nommées** : 18.5, 18.6, 241.1, 241.2, 241.5-241.7, 245.1, 245.2, 245.4-245.8
(sur Définitions 1.1-1.6, Théorèmes 18.1-18.4, 40.1, 241.3-241.4, 245.3, 245.9 — la chaîne la
plus longue du compendium, douze chapitres jusqu'à l'axiome). **Standing** : théorème
conditionnel ; **preprint non relu**. Les correspondances [77] (Stone/Hamiltonien), [78]
(énergie de repos $=$ masse, $c = 1$), [79] (déterminant $\det(I - uB)$ non-backtracking) sont
**par nom** : des objets finis de cette théorie qui portent les noms, pas des théorèmes de la
littérature établis ici.


## Exercice 3 — Lire une correspondance « par nom »

La correspondance [77] invoque le théorème de Stone : l'évolution d'un système fermé est un
groupe à un paramètre d'unitaires, exponentielle d'un générateur auto-adjoint — l'Hamiltonien —
dont les valeurs propres sont les niveaux d'énergie. Le chapitre fournit un objet fini qui
porte ce nom ; il ne dérive pas le théorème de Stone.

**Consigne.** En une phrase par ligne, dites pour chacune des trois correspondances [77], [78],
[79] ce qui est **établi** dans ce chapitre (l'objet fini, son niveau, son carré de Born) et ce
qui n'est **que nommé** (le théorème de la littérature). Le résultat attendu est un texte, pas
un calcul.


In [8]:
// Exercice 3 : a vous -- trois lignes, une par correspondance [77] [78] [79].
public static class Exercice3
{
    public static void Resoudre()
    {
        // TODO etudiant
        Console.WriteLine("Exercice a completer");
    }
}

Exercice3.Resoudre();

Exercice a completer


## Conclusion

Ce que ce carnet a **mesuré** : l'arithmétique exacte des quatre lignes du chapitre — la somme
$S_0 = 116 \cdot 36 = 4176$ (éq. 64), l'espace des enregistrements $N = 57$ dont la somme des
gains entiers retombe sur la face de Born $Z = 96$ en regard de la décomposition en secteurs de
l'éq. 58 (éq. 65), la valeur propre $4176$ de multiplicité 57 et les niveaux rationnels exacts
$1392/67$ et $4176/67$ — y compris le décodage des fractions écrasées du PDF (éq. 66) —, et le
corps électron $S = 67 \cdot 3 = 201$ aux carrés de Born 1 et 9 (éq. 67).

Ce qui reste **cité** : tout le census (309 lectures, six types de paires, 32851 coups, treize
corps, deux moteurs), la trace d'anneau $30 + 30\sqrt2$, le commutateur nul aux douze tests, la
construction de l'opérateur et du pas. Ce qui reste **ouvert** : le générateur debout n'égale
la somme à **aucune** des 309 lectures, le transport et son spectre **partent** aux 1692 liens,
et les prémices ouvertes nommées en §6 tiennent la chaîne de douze chapitres.

Avec le pas et son générateur en main, le chapitre suivant (Chaînes-06) somme sur les
enregistrements que le pas laisse — l'intégrale de chemin comme somme finie d'enregistrements,
premier résultat ouvert de la série.


## Références

- Wissner-Gross, A. (2026). *Chain Theory Working Draft* (compendium v0.1 working draft,
  preprint non relu), chapitre 44 « Energy and the Hamiltonian: the first-power operator and
  the step », p. 121-124. Copie archivée au gisement avant écriture.
- Correspondances citées par nom dans le chapitre : **[77]** théorème de Stone (groupe à un
  paramètre d'unitaires, générateur auto-adjoint, niveaux d'énergie) ; **[78]** énergie de
  repos d'un corps = sa masse à l'unité $c = 1$ ; **[79]** déterminant $\det(I - uB)$ de
  l'opérateur non-backtracking $B$ d'un graphe fini.
- Carnets de la série : [Chaînes-01](Chaines-01-Axiome-Intrication-Constante-Structure-Fine-Python.ipynb)
  (l'axiome et la constante de structure fine) ·
  [Chaînes-02](Chaines-02-Born-Unitarite-Incertitude-Python.ipynb)
  (Born, unitarité, incertitude) ·
  [Chaînes-03](Chaines-03-Mesure-Lavage-POVM-Python.ipynb)
  (la mesure — éq. 58, le même espace d'enregistrements) ·
  [Chaînes-04](Chaines-04-Indiscernabilite-Spin-Statistique-Python.ipynb)
  (l'indiscernabilité et le signe spin-statistiques).


## À retenir

1. **L'énergie est une somme à première puissance** : $S_0 = 116 \cdot 36 = 4176$ (éq. 64), la
   mesure de la toile multipliée par son produit hub — deux nombres **donnés** par la source, dont
   le carnet vérifie qu'ils rendent bien la somme annoncée.
2. **La valeur propre tombe sur chaque enregistrement** : $H\psi = 4176\,\psi$ sur un espace de
   $N = 57$ dimensions, donc une valeur propre de multiplicité 57. La construction de $H$ est
   citée ; l'arithmétique, non.
3. **Les niveaux sont des rationnels exacts** : $1392/67$ à l'unité hub et $4176/67$ à l'unité
   dock — exactement les fractions que le texte du PDF écrase en « 139267 » et « 417667 ». Le
   carnet les décode et vérifie l'égalité.
4. **Le corps électron** $S = 67 \cdot 3 = 201$ porte les carrés de Born 1 et 9 (éq. 67) ;
   l'additivité en rang de l'exercice 2 en est le contrôle.
5. **Le cœur physique reste cité** : le commutateur nul $[H, G_{\text{step}}] = 0$ aux douze
   tests de commutation, la construction de l'opérateur et celle du pas. Le carnet mesure
   l'arithmétique du chapitre, pas la physique qui la motive.


## Vérifiez votre compréhension

1. Comment se décompose $S_0$, et que vérifie le carnet à son sujet ?
2. Quelles fractions le PDF écrase-t-il, et quels rationnels le décodage rend-il ?
3. Pourquoi la valeur propre $4176$ est-elle de multiplicité 57 ?
4. Que vaut le corps électron, et quels sont ses carrés de Born ?

<details>
<summary>Réponses</summary>

1. $S_0 = 116 \cdot 36 = 4176$ : la mesure de la toile (116, ses deux sites qubits) multipliée par
   son produit hub (36). Les deux nombres sont **cités** ; le carnet vérifie que la règle de
   première puissance appliquée à eux rend la somme annoncée par l'éq. 64.
2. « 139267 » et « 417667 », qui se décodent en $1392/67$ (niveau à l'unité hub, $(4176/3)/67$) et
   $4176/67$ (niveau à l'unité dock) — les deux égalités sont testées avec le rationnel exact `Rat`.
3. Parce que l'espace des enregistrements de l'éq. 65 a $N = 24 + 18 + 12 + 3 = 57$ dimensions et
   que l'opérateur à première puissance y porte la somme 4176 **sur chaque enregistrement** :
   chaque enregistrement est une copie de la valeur propre.
4. $S = 67 \cdot 3 = 201$ (éq. 67) : niveau 1 à l'unité hub, niveau 3 à l'unité dock, donc les
   carrés de Born 1 et 9 — le carré du niveau. Le rang dock-frappé 67 et l'habillage hub 3
   restent cités.

</details>


## Pour aller plus loin

*Chaînes-06 — l'intégrale de chemin comme somme finie d'enregistrements*
prend le pas et son générateur là où ce carnet les laisse : il somme sur les enregistrements que
le pas laisse, et c'est le premier résultat ouvert de la série — la construction est citée, le
comptage est rejoué.

En amont, [Chaînes-04 — l'indiscernabilité et le signe spin-statistique](Chaines-04-Indiscernabilite-Spin-Statistique-Python.ipynb)
fournit les deux faces du signe d'échange dont la valeur propre d'ici hérite, et
[Chaînes-03 — la mesure](Chaines-03-Mesure-Lavage-POVM-Python.ipynb) le cadre de lecture
collective qui donne son sens à « un enregistrement = une copie ».

Hors de la série, [Jauge-01 — le théorème de Noether](../Jauge/Jauge-01-Theoreme-Noether-Python.ipynb)
est le contraste exact : la conservation y est un théorème dérivé, alors qu'ici le commutateur nul
qui l'exprime est **cité** aux douze tests. Le census du §5 (309 lectures, six types de paires,
32851 coups, treize corps, deux moteurs) est le point de reprise pour qui veut le vérifier.